In [ ]:
import numpy as np
from datetime import datetime, timezone

def get_seconds_of_day(time_str):
    # Parse the datetime from the string
    dt = datetime.strptime(time_str, '%y%m%d%H%M%S')
    
    # Get midnight of the same day
    midnight = datetime(year=dt.year, month=dt.month, day=dt.day)
    
    # Calculate the number of seconds since midnight
    seconds_since_midnight = (dt - midnight).seconds
    
    return seconds_since_midnight

def last_indices(s):
    # Find the indices of the last character of each number in the string
    indices = []
    position = 0
    # Split the string by whitespace to get each number as a separate element
    numbers = s.strip().split()

    # Iterate through the numbers to find the last index of each
    for number in numbers:
        # Find the end index of the current number
        position = s.find(number, position) + len(number) - 1
        indices.append(position)
        # Update the position to start search for the next number after this one
        position += 1

    # Print the resulting indices
    return indices

def parse_reflectivity(indices, data_str):
    s_list = list(data_str)
    
    # Iterate over the list of indices
    for index in indices:

        if len(s_list) <= index:
            break
        # Check if the character at the current index is a space
        if s_list[index] == ' ':
            s_list[index] = 'nan'
    
    # Convert the list of characters back to a string
    result = (''.join(s_list)).split(' ')
    del result[0]
    result = [item for item in result if item.strip()]
    while len(result) < len(indices)-1:
        result.append('nan')
    return result

def read_specific_lines(filename):
    # List of prefixes to look for
    prefixes = ('MRR', 'H', 'TF', 'z')
    
    lines = []
    # Open the file
    with open(filename, 'r') as file:
        # Loop through each line in the file
        for line in file:
            # Strip whitespace from the beginning and end of the line
            stripped_line = line.strip()
            # Check if the line starts with any of the specified prefixes
            if stripped_line.startswith(prefixes):
                # Print or process the line
                # print(stripped_line)
                lines.append(stripped_line)
        return lines

file_path = "/Users/Indhuja/Desktop/MRR/olympex_mrr01_20160115_pro.txt"
line = read_specific_lines(file_path)
skip_count = 1
temp = skip_count

i = 0

time = []
lat = []
lon = []
alt = []
ref = []
latitude = 47.970
longitude = -123.499
while i < len(line):
    if temp % skip_count != 0:
        i += 4
        temp += 1
        continue

    # Parse timestamp and convert to seconds of the day
    timestamp = line[i][4:16]
    delta_sec = get_seconds_of_day(timestamp)

    i += 1
    heights_line = line[i]
    heights = []
    for item in line[i].split():
        try:
            # Try converting each item to an integer
            height = int(item)
            heights.append(height)
        except ValueError:
            # Ignore any items that cannot be converted to an integer (non-numeric strings)
            continue

    i += 1
    tf_values = []
    for item in line[i].split():
        try:
            tf = float(item)
            tf_values.append(tf)
        except ValueError:
            continue

    adjusted_heights = [h * tf for h, tf in zip(heights, tf_values)]
    alt.append(adjusted_heights)

    i += 1
    reflect = parse_reflectivity(last_indices(heights_line), line[i])
    ref.append(reflect)

    # Append latitude and longitude repeated according to the number of heights
    lat.append(np.repeat(latitude, len(adjusted_heights)))
    lon.append(np.repeat(longitude, len(adjusted_heights)))
    time.append(np.repeat(delta_sec, len(adjusted_heights)))

    i += 1
    temp += 1

8
delta_sec  5
adjusted_heights  [1.278, 7.872000000000001, 24.264, 54.263999999999996, 97.86, 155.124, 223.39800000000002, 300.432, 385.83000000000004, 473.94000000000005, 562.254, 654.9119999999999, 739.206, 819.168, 890.4599999999999, 959.136, 1020.0, 1072.332, 1119.822, 1163.8799999999999, 1188.558, 1212.288, 1225.854, 1240.848, 1244.7, 1250.184, 1240.92, 1230.264, 1209.3, 1091.52, 778.2239999999999]
woowo  z   -28.78 -13.90  -8.58 -13.43                      -13.30 -16.78        -14.48  -6.40 -10.20  -7.67  -8.35 -21.63                                     -7.08  -8.27  -3.42         -3.69  -1.65  -9.95  -2.08   2.81   5.85
32
ref  [['-28.78', '-13.90', '-8.58', '-13.43', 'nan', 'nan', 'nan', '-13.30', '-16.78', 'nan', '-14.48', '-6.40', '-10.20', '-7.67', '-8.35', '-21.63', 'nan', 'nan', 'nan', 'nan', 'nan', '-7.08', '-8.27', '-3.42', 'nan', '-3.69', '-1.65', '-9.95', '-2.08', '2.81', '5.85']]
lat  [array([47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97,
       47.97

In [3]:
import pandas as pd
import numpy as np
import json

# Flatten lists
flat_time = [item for sublist in time for item in sublist]
flat_lat = [item for sublist in lat for item in sublist]
flat_lon = [item for sublist in lon for item in sublist]
flat_alt = [item for sublist in alt for item in sublist]
flat_ref = [float(item) for sublist in ref for item in sublist]


df = pd.DataFrame({
    'Time': flat_time,
    'Latitude': flat_lat,
    'Longitude': flat_lon,
    'Altitude': flat_alt,
    'Reflectivity': flat_ref
})

df = df.replace('nan', np.nan).dropna()
df = df.reset_index(drop=True)

def color_encode(data, min_value, max_value, opa=255, RGBA=True):
    num_colors = 100
    colors = np.linspace([0, 0, 1], [1, 0, 0], num_colors)
    if RGBA:
        colors = np.column_stack((colors, np.full(num_colors, opa)))
    normalized_values = (data - min_value) / (max_value - min_value)
    normalized_values = np.clip(normalized_values, 0, 1)
    color_indices = (normalized_values * (num_colors - 1)).astype(int)
    return colors[color_indices]


encoded_colors = color_encode(df['Reflectivity'], df['Reflectivity'].min(), df['Reflectivity'].max())

result_dict = {}

time = -1
index = -1
for idx, row in df.iterrows():
    if row['Time'] != time:
        time = row['Time']  
        index = index + 1;  
        result_dict[index] = {
            'id': row['Time'],
            'Lat': [row['Latitude']],
            'Lon': [row['Longitude']],
            'Alt': [],
            'Color': []
        }
       
    result_dict[index]['Lat'].append(row['Latitude'])
    result_dict[index]['Lon'].append(row['Longitude'])
    result_dict[index]['Alt'].append(row['Altitude'])
    result_dict[index]['Color'].append(encoded_colors[idx].tolist())



for idx, row in df.iterrows():
    result_dict[str(idx)] = {
        'id': row['Time'],
        'Lat': [row['Latitude']],  # As array
        'Lon': [row['Longitude']],  # As array
        'Alt': [row['Altitude']],  # As array
        'Color': encoded_colors[idx].tolist()  # Adding encoded colors
    }

print(result_dict)

with open('/Users/Indhuja/Downloads/mrr_reflectivity_data.json', 'w') as file:
    json.dump(result_dict, file, indent=4)

[1.278, 7.872000000000001, 24.264, 54.263999999999996, 97.86, 155.124, 223.39800000000002, 300.432, 385.83000000000004, 473.94000000000005, 562.254, 654.9119999999999, 739.206, 819.168, 890.4599999999999, 959.136, 1020.0, 1072.332, 1119.822, 1163.8799999999999, 1188.558, 1212.288, 1225.854, 1240.848, 1244.7, 1250.184, 1240.92, 1230.264, 1209.3, 1091.52, 778.2239999999999, 1.278, 7.872000000000001, 24.264, 54.263999999999996, 97.86, 155.124, 223.39800000000002, 300.432, 385.83000000000004, 473.94000000000005, 562.254, 654.9119999999999, 739.206, 819.168, 890.4599999999999, 959.136, 1020.0, 1072.332, 1119.822, 1163.8799999999999, 1188.558, 1212.288, 1225.854, 1240.848, 1244.7, 1250.184, 1240.92, 1230.264, 1209.3, 1091.52, 778.2239999999999]
62 62 62 62 62
{0: {'id': 5.0, 'Lat': [47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97, 47.97], 'Lon': [-123.499, -123.499, -123.499, -123.499, -123.499,